# `elixir-query` end-to-end demo

Each section below imports `elixir_query`, fetches data from one ELIXIR Core Data Resource via `eq.get(...)`, inspects the schema, and runs a small analysis. If anything errors or returns unexpected data the upstream adapter is fixed in the package itself.

Adapters covered in this pass: **UniProt**, **HGNC**, **ChEMBL**.

In [ ]:
import polars as pl
import elixir_query as eq

print(f"elixir-query version: {eq.__version__}")
print(f"databases registered: {len(eq.list_databases())}")
print(eq.list_databases())

## 1. UniProt — protein knowledgebase

Workflow:
1. Fetch a single canonical entry (P00533 = EGFR_HUMAN).
2. Inspect the schema and a sentinel value.
3. Run a small Lucene search and aggregate by length.

In [ ]:
# 1.1 Fetch a single UniProt entry by accession.
egfr = eq.get("uniprot", accession="P00533")
print(f"shape: {egfr.shape}")
print(f"columns: {egfr.columns}")
egfr

In [ ]:
# 1.2 Sentinel checks: the EGFR record should be human (organism 9606)
#     and 1210 amino acids long. UniProt humanises column names so we
#     locate them by substring match.
org_col = next(c for c in egfr.columns if "organism" in c.lower() and "id" in c.lower())
len_col = next(c for c in egfr.columns if c.lower() == "length")

org_id = int(egfr[org_col][0])
length = int(egfr[len_col][0])
print(f"P00533 organism_id = {org_id} (expected 9606)")
print(f"P00533 length      = {length} (expected 1210)")
assert org_id == 9606 and length == 1210, "UniProt schema drift detected!"

In [ ]:
# 1.3 Lucene search + small analysis: human reviewed insulin-related proteins,
#     ranked by sequence length. Verifies the cursor-paginated /search path.
hits = eq.get(
    "uniprot",
    query="gene:insulin AND organism_id:9606 AND reviewed:true",
    limit=20,
)
print(f"hits: {hits.height}")

length_col = next(c for c in hits.columns if c.lower() == "length")
entry_col = next(c for c in hits.columns if c.lower() == "entry")

(
    hits.with_columns(pl.col(length_col).cast(pl.Int32))
    .sort(length_col, descending=True)
    .select([entry_col, length_col])
    .head(10)
)

## 2. HGNC — gene nomenclature

Workflow:
1. Fetch the BRCA2 record by approved symbol.
2. Verify the canonical HGNC ID `HGNC:1101`.
3. Free-text search for "BRCA" — confirm both BRCA1 and BRCA2 appear.

In [ ]:
# 2.1 Single-gene lookup by approved HGNC symbol.
brca2 = eq.get("hgnc", symbol="BRCA2")
print(f"shape: {brca2.shape}")
brca2.select(["hgnc_id", "symbol", "name", "locus_type", "location", "entrez_id"])

In [ ]:
# 2.2 Sentinel: BRCA2 must have HGNC:1101 and be a protein-coding gene.
hgnc_id = brca2["hgnc_id"][0]
locus_type = brca2["locus_type"][0]
print(f"BRCA2 hgnc_id    = {hgnc_id} (expected 'HGNC:1101')")
print(f"BRCA2 locus_type = {locus_type!r}")
assert hgnc_id == "HGNC:1101", "HGNC schema or data drift!"
assert locus_type == "gene with protein product"

In [ ]:
# 2.3 Free-text search: count BRCA-prefixed genes by locus group.
brca = eq.get("hgnc", search="BRCA")
print(f"BRCA hits: {brca.height}")

(
    brca.filter(pl.col("symbol").str.starts_with("BRCA"))
    .group_by("locus_group")
    .agg(pl.len().alias("count"), pl.col("symbol").alias("symbols"))
    .sort("count", descending=True)
)

## 3. ChEMBL — bioactive molecules

Workflow:
1. Fetch CHEMBL25 (aspirin) — verify pref_name and a molecular property.
2. Filtered list with pagination — pull the first page of approved drugs (`max_phase=4`).
3. Aggregate phase-4 drugs by `molecule_type`.

In [ ]:
# 3.1 Fetch a single molecule (CHEMBL25 = aspirin).
aspirin = eq.get("chembl", molecule_chembl_id="CHEMBL25")
print(f"shape: {aspirin.shape}")

cols_of_interest = [
    c for c in ["molecule_chembl_id", "pref_name", "molecule_type", "max_phase",
                "first_approval", "molecule_properties"]
    if c in aspirin.columns
]
aspirin.select(cols_of_interest)

In [ ]:
# 3.2 Sentinel: aspirin is approved (max_phase=4), pref_name='ASPIRIN'.
chembl_id = aspirin["molecule_chembl_id"][0]
pref_name = aspirin["pref_name"][0]
print(f"CHEMBL25 molecule_chembl_id = {chembl_id} (expected 'CHEMBL25')")
print(f"CHEMBL25 pref_name          = {pref_name} (expected 'ASPIRIN')")
assert chembl_id == "CHEMBL25"
assert pref_name == "ASPIRIN"

In [ ]:
# 3.3 Filtered search: small molecules at max_phase=4 (approved).
#     Exercises the page_meta.next pagination path.
phase4 = eq.get(
    "chembl",
    filters={"max_phase": "4", "molecule_type": "Small molecule"},
    limit=50,
)
print(f"phase-4 small molecules pulled: {phase4.height}")
phase4.select(["molecule_chembl_id", "pref_name", "molecule_type", "max_phase"]).head(10)

In [ ]:
# 3.4 Cache check: a second identical call should return the same frame
#     and not re-hit the network.
import time

t0 = time.perf_counter()
again = eq.get("chembl", molecule_chembl_id="CHEMBL25")
elapsed = time.perf_counter() - t0
print(f"second call took {elapsed*1000:.1f} ms (cache hit)")
assert again.equals(aspirin)